# CarGurus Multiple Pages Scraper

This notebook mirrors the `TrueCar_Multiple.ipynb` flow for CarGurus. It is currently tuned for used cars within 25 miles of Boston, MA:

1. collect vehicle cards from CarGurus search result pages,
2. save the listing links/metadata to JSON and CSV,
3. optionally visit each detail page for richer fields,
4. save the detail dataset to JSON and CSV.

CarGurus may reject plain `requests` traffic. If that happens, open the search URL in a browser, save the page HTML into `data/`, and set `LOCAL_HTML_PATHS` in the final cell.


In [1]:
from bs4 import BeautifulSoup
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse
import json
import os
import random
import re
import time

import pandas as pd
import requests
from tqdm import tqdm

try:
    from curl_cffi import requests as curl_requests
except ImportError:
    curl_requests = None


In [2]:
BASE_URL = "https://www.cargurus.com"
DATA_DIR = "./data"

SEARCH_ZIP = "02108"  # Downtown Boston, MA
SEARCH_DISTANCE_MILES = 25
SEARCH_SORT_TYPE = "BEST_MATCH"
SEARCH_SORT_DIRECTION = "ASC"
SEARCH_INVENTORY_TYPES = "USED"
RANDOM_SEED = None  # Set to an integer for repeatable delay timing while debugging.
DEFAULT_RETRY_DELAY_SECONDS = (20, 45)

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)

SEARCH_URL = (
    f"https://www.cargurus.com/search?"
    f"distance={SEARCH_DISTANCE_MILES}"
    f"&sortDirection={SEARCH_SORT_DIRECTION}"
    f"&sortType={SEARCH_SORT_TYPE}"
    f"&zip={SEARCH_ZIP}"
    f"&inventoryTypes={SEARCH_INVENTORY_TYPES}"
)

HEADERS = {
    "User-Agent": os.getenv(
        "CARGURUS_USER_AGENT",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": os.getenv("CARGURUS_ACCEPT_LANGUAGE", "en-US,en;q=0.9"),
    "Accept-Encoding": "gzip, deflate",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1",
}


In [3]:
def request_headers(referer=None):
    headers = dict(HEADERS)
    cookie = os.getenv("CARGURUS_COOKIE")
    if cookie:
        headers["Cookie"] = cookie
    if referer:
        headers["Referer"] = referer
    return headers


def sleep_for_delay(delay_seconds, label="delay"):
    if delay_seconds in [None, 0, (0, 0)]:
        return 0
    if isinstance(delay_seconds, tuple):
        low, high = delay_seconds
        seconds = random.uniform(min(low, high), max(low, high))
    else:
        seconds = float(delay_seconds)
    if seconds > 0:
        print(f"{label}: sleeping {seconds:.1f} seconds")
        time.sleep(seconds)
    return seconds


def get_soup(url, delay_seconds=(8, 25), referer=BASE_URL, retry_delay_seconds=DEFAULT_RETRY_DELAY_SECONDS):
    sleep_for_delay(delay_seconds, label="request delay")
    headers = request_headers(referer=referer)
    response = requests.get(url, headers=headers, timeout=30)

    if response.status_code in {403, 406, 429} and curl_requests is not None:
        print(f"requests was blocked with HTTP {response.status_code}; retrying with curl_cffi browser impersonation.")
        sleep_for_delay(retry_delay_seconds, label="retry backoff")
        response = curl_requests.get(
            url,
            headers=headers,
            impersonate=os.getenv("CARGURUS_IMPERSONATE", "chrome124"),
            timeout=30,
        )

    if response.status_code in {403, 406, 429}:
        print(f"Blocked with HTTP {response.status_code} for {url}")
        print("Try a legitimate CARGURUS_COOKIE / CARGURUS_USER_AGENT from your browser session, reduce volume, or parse saved HTML with LOCAL_HTML_PATHS.")
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def clean_text(value):
    if value is None:
        return None
    return re.sub(r"\s+", " ", str(value)).strip() or None


def parse_money(value):
    if value is None:
        return None
    match = re.search(r"\$?\s*([\d,]+(?:\.\d+)?)", str(value))
    return float(match.group(1).replace(",", "")) if match else None


def parse_integer(value):
    if value is None:
        return None
    match = re.search(r"\d[\d,]*", str(value))
    if not match:
        return None
    digits = match.group(0).replace(",", "")
    return int(digits) if digits else None


def slug_label(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


In [4]:
def split_title(title):
    if not title:
        return [None, None, None, None]
    clean = re.sub(r"^(Used|New|Certified)\s+", "", title.strip(), flags=re.IGNORECASE)
    parts = clean.split()
    if len(parts) < 3 or not parts[0].isdigit():
        return [None, None, None, None]
    year = int(parts[0])
    make = parts[1]
    model = parts[2]
    trim = " ".join(parts[3:]) if len(parts) > 3 else None
    return [year, make, model, trim]


def search_page_url(search_url, page):
    if page <= 1:
        return search_url

    parsed = urlparse(search_url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query["page"] = str(page)
    return urlunparse(parsed._replace(query=urlencode(query), fragment=parsed.fragment))


def first_text(node, selectors):
    for selector in selectors:
        element = node.select_one(selector)
        if element:
            text = clean_text(element.get_text(" ", strip=True))
            if text:
                return text
    return None


def first_attr(node, selectors, attr):
    for selector in selectors:
        element = node.select_one(selector)
        if element and element.get(attr):
            return element.get(attr)
    return None


def looks_like_vehicle_url(href):
    if not href:
        return False
    return any(token in href for token in ["/details/", "/Cars/", "/car/", "/listing/", "listing="])


In [5]:
def iter_json_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_json_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_json_objects(child)


def load_json_from_script(script):
    text = script.string or script.get_text("", strip=False)
    if not text:
        return None
    text = text.strip()
    if not text:
        return None

    if script.get("type") == "application/ld+json":
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return None

    if script.get("id") == "__NEXT_DATA__" or script.get("type") == "application/json":
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return None

    remix_prefix = "window.__remixContext = "
    if remix_prefix in text:
        try:
            start = text.index(remix_prefix) + len(remix_prefix)
            return json.JSONDecoder().raw_decode(text[start:])[0]
        except (ValueError, json.JSONDecodeError):
            return None

    if not any(token in text for token in ["listingId", "listingTitle", "vehicleIdentificationNumber", "priceData"]):
        return None

    # Some pages hydrate state as JavaScript assignments. This conservative fallback
    # only attempts to load the first object-looking block.
    match = re.search(r"({.*})", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None


def nested_get(record, *paths):
    for path in paths:
        current = record
        for part in path:
            if isinstance(current, dict):
                current = current.get(part)
            else:
                current = None
                break
        if current not in [None, "", []]:
            return current
    return None


def normalize_listing_record(record, search_url, source="embedded_json"):
    if record.get("type", "").startswith("LISTING") and isinstance(record.get("data"), dict):
        record = record["data"]
    else:
        has_listing_shape = any(key in record for key in ["listingTitle", "priceData", "sellerData"])
        has_vehicle_identity = any(key in record for key in ["id", "listingId", "vin", "ontologyData"])
        if not (has_listing_shape and has_vehicle_identity):
            return None

    listing_id = nested_get(record, ["id"], ["listingId"], ["listing", "id"], ["listing", "listingId"])
    vin = nested_get(record, ["vin"], ["vehicleIdentificationNumber"], ["vehicle", "vin"], ["vdp", "vin"])
    title = nested_get(record, ["listingTitle"], ["title"], ["vehicleTitle"], ["heading"], ["name"])

    year = parse_integer(nested_get(record, ["year"], ["carYear"], ["ontologyData", "carYear"], ["vehicle", "year"], ["modelYear"]))
    make = nested_get(record, ["makeName"], ["make"], ["ontologyData", "makeName"], ["vehicle", "make"], ["vehicle", "makeName"])
    model = nested_get(record, ["modelName"], ["model"], ["ontologyData", "modelName"], ["vehicle", "model"], ["vehicle", "modelName"])
    trim = nested_get(record, ["trimName"], ["trim"], ["ontologyData", "trimName"], ["vehicle", "trim"], ["vehicle", "trimName"])
    if not title and year and make and model:
        title = " ".join(str(part) for part in [year, make, model, trim] if part)
    if title and not (year and make and model):
        parsed_year, parsed_make, parsed_model, parsed_trim = split_title(title)
        year = year or parsed_year
        make = make or parsed_make
        model = model or parsed_model
        trim = trim or parsed_trim

    href = nested_get(record, ["url"], ["detailUrl"], ["vdpUrl"], ["listingUrl"], ["vehicle", "url"])
    detail_url = urljoin(BASE_URL, href) if href else None
    if not detail_url and listing_id:
        detail_url = f"{BASE_URL}/details/{listing_id}"

    price = nested_get(record, ["price"], ["priceString"], ["priceData", "localizedTotalPrice"], ["priceData", "localizedPrice"], ["priceData", "totalPrice"], ["priceData", "current"], ["salePrice"], ["listPrice"])
    mileage = nested_get(record, ["mileage"], ["mileageString"], ["localizedMileage"], ["mileageData", "value"], ["miles"], ["vehicle", "mileage"])
    dealer = nested_get(record, ["dealerName"], ["sellerName"], ["sellerData", "serviceProviderName"], ["dealer", "name"], ["seller", "name"])
    location = nested_get(record, ["location"], ["sellerData", "displayLocation"], ["dealer", "city"], ["seller", "city"])
    exterior = nested_get(record, ["exteriorColorData", "localized"], ["exteriorColor"], ["exterior"])
    interior = nested_get(record, ["interiorColorData", "localized"], ["interiorColor"], ["interior"])
    image = nested_get(record, ["pictureData", "url"], ["imageUrl"], ["photoUrl"])

    row = {
        "url": detail_url,
        "listing_id": listing_id,
        "vin": vin,
        "title": clean_text(title),
        "year": year,
        "make": clean_text(make),
        "model": clean_text(model),
        "trim": clean_text(trim),
        "list_price": parse_money(price),
        "list_price_displayed": clean_text(price),
        "odometer_miles": parse_integer(mileage),
        "mileage_displayed": clean_text(mileage),
        "dealer_info": clean_text(dealer),
        "location": clean_text(location),
        "exterior": clean_text(exterior),
        "interior": clean_text(interior),
        "drivetrain": clean_text(nested_get(record, ["localizedDrivetrain"])),
        "engine": clean_text(nested_get(record, ["localizedEngineName"])),
        "transmission": clean_text(nested_get(record, ["localizedTransmission"])),
        "fuel_type": clean_text(nested_get(record, ["fuelData", "localizedType"])),
        "stock_number": clean_text(nested_get(record, ["stockNumber"])),
        "days_on_market": parse_integer(nested_get(record, ["daysOnMarket"])),
        "distance_miles": nested_get(record, ["distance"]),
        "deal_rating": clean_text(nested_get(record, ["dealRating"])),
        "features": "; ".join(nested_get(record, ["vehicleFeatures"]) or []) or None,
        "listing_image_url": image,
        "source_name": "CarGurus",
        "source_parser": source,
        "return_to": search_url,
    }
    return row if row["url"] or row["listing_id"] or row["vin"] else None


def parse_embedded_listing_data(soup, search_url):
    rows = []
    seen = set()
    for script in soup.find_all("script"):
        data = load_json_from_script(script)
        if data is None:
            continue
        for record in iter_json_objects(data):
            row = normalize_listing_record(record, search_url)
            if not row:
                continue
            key = row.get("url") or row.get("listing_id") or row.get("vin")
            if key and key not in seen:
                rows.append(row)
                seen.add(key)
    return rows


In [6]:
def listing_card_containers(soup):
    selectors = [
        "[data-testid='srp-listing-tile']",
        "[data-testid*='listing']",
        "[data-test*='listing']",
        "[data-cg-ft='car-blade']",
        ".cg-listing-search-result",
        ".listing-card",
        "article",
    ]
    cards = []
    seen = set()
    for selector in selectors:
        for card in soup.select(selector):
            text = clean_text(card.get_text(" ", strip=True)) or ""
            href = first_attr(card, ["a[href]"], "href")
            if len(text) < 20 or not looks_like_vehicle_url(href):
                continue
            key = id(card)
            if key not in seen:
                cards.append(card)
                seen.add(key)
    return cards


def parse_definition_details(card):
    details = {}
    for label in card.select("dt"):
        value = label.find_next_sibling("dd")
        key = clean_text(label.get_text(" ", strip=True))
        val = clean_text(value.get_text(" ", strip=True)) if value else None
        if key and val:
            details[slug_label(key)] = val
    return details


def parse_listing_card(card, search_url):
    href = first_attr(card, ["a[href*='/details/']", "a[href*='listing=']", "a[href*='/Cars/']", "a[href*='/listing/']", "a[href]"], "href")
    if not looks_like_vehicle_url(href):
        return None

    detail_url = urljoin(BASE_URL, href)
    title = first_text(card, ["[data-testid='srp-listing-blade-title']", "[data-testid='srp-tile-listing-title'] h5", "h1", "h2", "h3", "[data-testid*='title']", "[data-test*='title']", "a[href]"])
    year, make, model, trim = split_title(title)
    details = parse_definition_details(card)
    year = year or parse_integer(details.get("year"))
    make = make or details.get("make")
    model = model or details.get("model")
    trim = trim or first_text(card, ["[data-testid='srp-tile-listing-title'] p"])

    listing_id = None
    match = re.search(r"/details/(\d+)|listing[=/](\d+)|listing=(\d+)", href)
    if match:
        listing_id = next(group for group in match.groups() if group)

    price_text = first_text(card, ["[data-testid='srp-tile-price']", "[data-testid*='price']", "[data-test*='price']", ".price", "[class*='price']"])
    mileage_text = first_text(card, ["[data-testid*='mileage']", "[data-test*='mileage']", "[class*='mileage']", "[class*='Mileage']"])
    if not mileage_text:
        mileage_text = details.get("mileage")
    dealer_text = first_text(card, ["[data-testid*='dealer']", "[data-test*='dealer']", "[class*='dealer']", "[class*='seller']"])
    image_url = first_attr(card, ["img[src]", "img[data-src]"], "src") or first_attr(card, ["img[data-src]"], "data-src")

    row = {
        "url": detail_url,
        "listing_id": listing_id,
        "vin": details.get("vin") or first_attr(card, ["[data-vin]"], "data-vin"),
        "title": title,
        "year": year,
        "make": make,
        "model": model,
        "trim": trim,
        "list_price": parse_money(price_text),
        "list_price_displayed": price_text,
        "odometer_miles": parse_integer(mileage_text),
        "mileage_displayed": mileage_text,
        "dealer_info": dealer_text,
        "location": first_text(card, ["[data-testid='LocationSection-firstLine']", "[data-testid*='location']", "[class*='location']"]),
        "distance_miles": parse_integer(first_text(card, ["[data-testid='LocationSection-secondLine']"])),
        "body_type": details.get("body_type"),
        "drivetrain": details.get("drivetrain"),
        "engine": details.get("engine"),
        "exterior": details.get("exterior_color"),
        "interior": details.get("interior_color"),
        "fuel_type": details.get("fuel_type"),
        "transmission": details.get("transmission"),
        "listing_image_url": urljoin(BASE_URL, image_url) if image_url else None,
        "source_name": "CarGurus",
        "source_parser": "html_card",
        "return_to": search_url,
    }
    return row


def parse_cards_from_soup(soup, search_url):
    rows = parse_embedded_listing_data(soup, search_url)

    for card in listing_card_containers(soup):
        row = parse_listing_card(card, search_url)
        if row:
            rows.append(row)

    if not rows:
        print("No listing records found in this page.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    for col in ["listing_id", "vin", "url"]:
        if col in df.columns:
            has_value = df[col].notna() & (df[col].astype(str).str.len() > 0)
            with_value = df[has_value].drop_duplicates(subset=[col])
            without_value = df[~has_value]
            df = pd.concat([with_value, without_value], ignore_index=True)
    return df.reset_index(drop=True)


In [7]:
def collect_search_results(search_url, max_pages=1, delay_seconds=(8, 25), retry_delay_seconds=DEFAULT_RETRY_DELAY_SECONDS):
    frames = []
    for page in tqdm(range(1, max_pages + 1), desc="CarGurus search pages"):
        page_url = search_page_url(search_url, page)
        try:
            soup = get_soup(page_url, delay_seconds=delay_seconds, referer=BASE_URL, retry_delay_seconds=retry_delay_seconds)
        except requests.RequestException as exc:
            print(f"Request failed for page {page}: {exc}")
            print("If this URL opens in your browser, save the page HTML and set LOCAL_HTML_PATHS in the final cell.")
            break

        page_df = parse_cards_from_soup(soup, page_url)
        if page_df.empty:
            break
        frames.append(page_df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["url"]).reset_index(drop=True)


def collect_search_results_from_html(html_paths, search_url):
    frames = []
    for html_path in html_paths:
        html_path = str(html_path).strip()
        if not html_path:
            continue
        with open(html_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")
        page_df = parse_cards_from_soup(soup, search_url)
        if not page_df.empty:
            page_df["local_html_path"] = html_path
            frames.append(page_df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["url"]).reset_index(drop=True)


def save_listing_cards(df, output_prefix, data_dir=DATA_DIR):
    os.makedirs(data_dir, exist_ok=True)
    json_path = os.path.join(data_dir, f"{output_prefix}_links.json")
    csv_path = os.path.join(data_dir, f"{output_prefix}_links.csv")
    records = df.to_dict(orient="records")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)
    df.to_csv(csv_path, index=False)
    print(f"Saved {len(df)} listing cards")
    print(f"- {json_path}")
    print(f"- {csv_path}")
    return json_path, csv_path


In [8]:
def split_description_and_options(description):
    description = clean_text(description)
    if not description:
        return None, None
    marker = "[!@@Additional Info@@!]"
    if marker not in description:
        return description, None
    notes, options = description.split(marker, 1)
    option_items = [clean_text(item) for item in options.split(",")]
    option_items = [item for item in option_items if item]
    return clean_text(notes), "; ".join(dict.fromkeys(option_items)) or None


def first_detail_route(data):
    if not isinstance(data, dict):
        return None
    loader_data = nested_get(data, ["state", "loaderData"])
    if not isinstance(loader_data, dict):
        return None
    for key, value in loader_data.items():
        if "details" in str(key) and isinstance(value, dict) and (value.get("oldData") or value.get("data")):
            return value
    return None


def parse_detail_embedded_data(soup, car_data):
    for script in soup.find_all("script"):
        data = load_json_from_script(script)
        route = first_detail_route(data)
        if not route:
            continue

        old_data = route.get("oldData") or {}
        data_block = route.get("data") or {}
        listing = old_data.get("listing") or data_block.get("listing") or {}
        seller = data_block.get("seller") or old_data.get("seller") or {}
        pricing = route.get("pricing") or {}
        history = route.get("history") or {}
        review_summary = route.get("dealerReviewSummary") or {}

        seller_notes, options = split_description_and_options(listing.get("description"))
        pictures = listing.get("pictures") or []
        first_picture = pictures[0] if pictures and isinstance(pictures[0], dict) else {}
        address = seller.get("address") or {}
        rating = seller.get("rating") or {}

        updates = {
            "listing_id": listing.get("id") or nested_get(route, ["plxFormData", "listingId"]),
            "title": listing.get("listingTitle") or listing.get("listingTitleOnly"),
            "year": parse_integer(listing.get("year")),
            "make": listing.get("makeName"),
            "model": listing.get("modelName"),
            "trim": listing.get("trimName"),
            "body_type": listing.get("bodyTypeName"),
            "vin": listing.get("vin"),
            "stock_number": listing.get("stockNumber"),
            "list_price": parse_money(pricing.get("pricePlusFees") or pricing.get("basePrice") or listing.get("price")),
            "list_price_displayed": pricing.get("displayPrice") or listing.get("priceString"),
            "expected_price": parse_money(pricing.get("expectedPrice") or listing.get("expectedPrice")),
            "price_differential": parse_money(pricing.get("priceDifferential") or listing.get("priceDifferential")),
            "deal_rating": pricing.get("dealRating") or listing.get("dealRatingKey"),
            "odometer_miles": parse_integer(listing.get("mileage")),
            "mileage_displayed": listing.get("mileageString"),
            "exterior": listing.get("localizedExteriorColor"),
            "interior": listing.get("localizedInteriorColor"),
            "fuel_type": listing.get("localizedFuelType"),
            "engine": listing.get("localizedEngineDisplayName"),
            "drivetrain": listing.get("localizedDriveTrain"),
            "transmission": listing.get("localizedTransmission"),
            "city_mpg": parse_integer(listing.get("cityFuelEconomy")),
            "combined_mpg": parse_integer(listing.get("localizedCombinedFuelEconomy")),
            "seller_notes": seller_notes,
            "options_and_features": options,
            "days_at_dealer": parse_integer(history.get("daysAtDealer")),
            "days_on_cargurus": parse_integer(history.get("daysOnCarGurus")),
            "saved_count": parse_integer(history.get("savedCount") or listing.get("savedCount")),
            "dealer_name": seller.get("name"),
            "dealer_phone": seller.get("phoneNumberString") or seller.get("phoneNumberIntl"),
            "dealer_rating": rating.get("averageRating") or review_summary.get("averageRating"),
            "dealer_review_count": rating.get("reviewCount") or review_summary.get("reviewCount"),
            "dealer_street": address.get("street"),
            "dealer_city": address.get("city"),
            "dealer_state": address.get("region"),
            "dealer_zip": address.get("postalCode"),
            "location": address.get("cityRegion"),
            "distance_miles": listing.get("roundedDistance") or listing.get("distance"),
            "listing_image_url": first_picture.get("url") or first_picture.get("largeUrl"),
            "picture_count": len(pictures) if pictures else None,
            "source_parser_detail": "remix_json",
        }

        for key, value in updates.items():
            if value not in [None, "", []]:
                car_data[key] = value
        return car_data

    return car_data


def parse_detail_page(soup, car_data):
    car_data = parse_detail_embedded_data(soup, car_data)

    for script in soup.find_all("script", type="application/ld+json"):
        data = load_json_from_script(script)
        for record in iter_json_objects(data):
            if record.get("@type") in {"Vehicle", "Car", "Product"}:
                for key, value in record.items():
                    if isinstance(value, (str, int, float)) and value not in [None, ""]:
                        car_data[f"schema_{slug_label(key)}"] = value

    for label in soup.select("dt, th"):
        value = label.find_next_sibling(["dd", "td"])
        key = clean_text(label.get_text(" ", strip=True))
        val = clean_text(value.get_text(" ", strip=True)) if value else None
        if key and val:
            car_data[f"detail_{slug_label(key)}"] = val

    text = clean_text(soup.get_text(" ", strip=True)) or ""
    for label, pattern in {
        "vin": r"\bVIN\b\s*:?\s*([A-HJ-NPR-Z0-9]{11,17})",
        "stock_number": r"\bStock(?:\s*#| Number)?\b\s*:?\s*([A-Za-z0-9-]+)",
    }.items():
        if not car_data.get(label):
            match = re.search(pattern, text, flags=re.IGNORECASE)
            if match:
                car_data[label] = match.group(1)

    if not car_data.get("seller_notes"):
        description_header = soup.find(["h2", "h3"], string=re.compile(r"Dealer.?s description|Description", re.IGNORECASE))
        if description_header:
            container = description_header.find_parent()
            if container:
                for candidate in container.find_all(["p", "div"], limit=8):
                    nearby = clean_text(candidate.get_text(" ", strip=True))
                    if nearby and len(nearby) > 80 and "function(" not in nearby:
                        car_data["seller_notes"] = nearby
                        break

    return car_data


def extract_car_details(entry, delay_seconds=(3, 10), retry_delay_seconds=DEFAULT_RETRY_DELAY_SECONDS):
    car_data = entry.copy()
    link = entry.get("url")
    if not link:
        car_data["scrape_error"] = "Missing detail URL"
        return car_data

    try:
        soup = get_soup(link, delay_seconds=delay_seconds, referer=entry.get("return_to") or BASE_URL, retry_delay_seconds=retry_delay_seconds)
    except requests.RequestException as exc:
        car_data["scrape_error"] = str(exc)
        return car_data

    try:
        return parse_detail_page(soup, car_data)
    except Exception as exc:
        car_data["scrape_error"] = str(exc)
        return car_data


def scrape_all_car_details(json_path, output_json, output_csv, delay_seconds=(3, 10), retry_delay_seconds=DEFAULT_RETRY_DELAY_SECONDS):
    with open(json_path, "r", encoding="utf-8") as f:
        car_links = json.load(f)

    results = []
    for entry in tqdm(car_links, desc="Scraping car details"):
        results.append(extract_car_details(entry, delay_seconds=delay_seconds, retry_delay_seconds=retry_delay_seconds))

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print("Saved detail data to:")
    print(f"- {output_csv}")
    print(f"- {output_json}")
    return df


In [9]:
OUTPUT_PREFIX = "cargurus_boston_ma_25mi_used"

# First trial: one search page and no detail-page fan-out yet.
# If live requests are blocked, save one or more browser pages as HTML and list them here.
MAX_PAGES = 1
SCRAPE_DETAILS = False
LOCAL_HTML_PATHS = []  # Example: [r"data/cargurus_boston_ma_25mi_used_page1.html"]

# Random delay ranges in seconds. Use tuples so every request gets a different pause.
LISTING_DELAY_SECONDS = (10, 30)
DETAIL_DELAY_SECONDS = (5, 15)
RETRY_DELAY_SECONDS = (25, 60)

if LOCAL_HTML_PATHS:
    cards_df = collect_search_results_from_html(LOCAL_HTML_PATHS, SEARCH_URL)
else:
    cards_df = collect_search_results(
        SEARCH_URL,
        max_pages=MAX_PAGES,
        delay_seconds=LISTING_DELAY_SECONDS,
        retry_delay_seconds=RETRY_DELAY_SECONDS,
    )

links_json, links_csv = save_listing_cards(cards_df, OUTPUT_PREFIX)

if SCRAPE_DETAILS and not cards_df.empty:
    details_df = scrape_all_car_details(
        links_json,
        output_json=f"{DATA_DIR}/{OUTPUT_PREFIX}_details.json",
        output_csv=f"{DATA_DIR}/{OUTPUT_PREFIX}_details.csv",
        delay_seconds=DETAIL_DELAY_SECONDS,
        retry_delay_seconds=RETRY_DELAY_SECONDS,
    )
else:
    details_df = pd.DataFrame()

cards_df.head()


CarGurus search pages:   0%|          | 0/1 [00:00<?, ?it/s]

request delay: sleeping 19.8 seconds
requests was blocked with HTTP 406; retrying with curl_cffi browser impersonation.
retry backoff: sleeping 28.7 seconds


CarGurus search pages: 100%|██████████| 1/1 [00:50<00:00, 50.02s/it]

Saved 22 listing cards
- ./data\cargurus_boston_ma_25mi_used_links.json
- ./data\cargurus_boston_ma_25mi_used_links.csv


,url,listing_id,vin,title,year,make,model,trim,list_price,list_price_displayed,...,stock_number,days_on_market,distance_miles,deal_rating,features,listing_image_url,source_name,source_parser,return_to,body_type
0,https://www.cargurus.com/details/448463590,448463590,19XFL1H85PE011451,2023 Honda Civic Hatchback,2023,Honda,Civic Hatchback,Sport Touring FWD,28985.0,"$28,985",...,79693A,36.0,17.382736,FAIR_PRICE,Leather Seats; Sunroof/Moonroof; Navigation Sy...,https://static.cargurus.com/images/forsale/202...,CarGurus,embedded_json,https://www.cargurus.com/search?distance=25&so...,NaN
1,https://www.cargurus.com/details/450861814,450861814,5XYP3DGC8SG595478,2025 Kia Telluride,2025,Kia,Telluride,EX AWD,41498.0,"$41,498",...,HKBSSG595478,11.0,11.638669,FAIR_PRICE,Leather Seats; Sunroof/Moonroof; Navigation Sy...,https://static.cargurus.com/images/forsale/202...,CarGurus,embedded_json,https://www.cargurus.com/search?distance=25&so...,NaN
2,https://www.cargurus.com/details/443429527,443429527,JF2SJAEC8JH484521,2018 Subaru Forester,2018,Subaru,Forester,2.5i Premium,16494.0,"$16,494",...,23628,91.0,22.998554,GOOD_PRICE,Sunroof/Moonroof; Power Mirror Package; Alloy ...,https://static.cargurus.com/images/forsale/202...,CarGurus,embedded_json,https://www.cargurus.com/search?distance=25&so...,NaN
3,https://www.cargurus.com/details/445619031,445619031,SALCP2BG3HH646878,2017 Land Rover Discovery Sport,2017,Land Rover,Discovery Sport,SE,10642.0,"$10,642",...,646878,67.0,6.058588,FAIR_PRICE,Sport Package; Leather Seats; Power Package; N...,https://static.cargurus.com/images/forsale/202...,CarGurus,embedded_json,https://www.cargurus.com/search?distance=25&so...,NaN
4,https://www.cargurus.com/details/449661769,449661769,KM8K6CAA6KU336789,2019 Hyundai Kona,2019,Hyundai,Kona,SEL AWD,11444.0,"$11,444",...,U336789,24.0,2.711034,FAIR_PRICE,Alloy Wheels; Bluetooth; Backup Camera; Blind ...,https://static.cargurus.com/images/forsale/202...,CarGurus,embedded_json,https://www.cargurus.com/search?distance=25&so...,NaN
